# Enfoque con Embeddings
Todo a embeddings (queries, sinopsis + año + director + keywords) --> 5 con mas similtud coseno (comparando queries vs sinopsis + año + director + keywords)

La estrategia consiste en representar tanto las películas como las preferencias de cada usuario en un espacio vectorial común, y recomendar las películas cuyo vector sea más similar al perfil del usuario.

1. unificar texto
2. embeddings con w2v o sentence transformer sobre texto y queries
3. similitud coseno text vs queries

In [1]:
!pip install datasets
!pip install sentence-transformers
!pip install gensim
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 41.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 15.3 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
import pandas as pd
import re       # libreria de expresiones regulares
import string   # libreria de cadena de caracteres
from unidecode import unidecode
import numpy as np
import multiprocessing

# mdoelos
from gensim.models.phrases import Phrases, Phraser
from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer


from sklearn.metrics.pairwise import cosine_similarity

## Carga de datasets

Traemos el dataset de sinopsis de peliculas de IMDb desde Hugging Face

In [3]:
df_pelis = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/peliculas_limpio.csv")

Traemos el dataset proporcionado con la información de los usuarios y sus respectivas queries

In [4]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/usuarios.csv")

In [5]:
usuarios

,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
0,U01,Valentina,definido,Durmiendo con su enemigo,Más allá de la muerte,Desaparecida,¡Olvídate de mí!,The Crazies,Quiero una película donde una mujer enfrenta u...
1,U02,Rodrigo,definido,Adiós Bafana,Mi pie izquierdo,L.A. Confidential,Érase una vez en América,El juego del halcón,Busco algo basado en hechos reales sobre corru...
2,U03,Camila,definido,Los padres de él,Mamá a la fuerza,Norbit,Elizabethtown,My Sassy Girl,Una comedia donde la relación entre dos person...
3,U04,Tomás,definido,Matrix,Fahrenheit 451,¡Olvídate de mí!,X-Men,El único,Algo que haga pensar sobre qué es real y qué e...
4,U05,Lucía,definido,La novia cadáver,Spirit: El corcel indomable,Las aventuras de Peabody y Sherman,Los Increíbles,Steamboy,Animación donde el protagonista lucha por su l...
5,U06,Martín,definido,Bienvenidos a Collinwood,El gran golpe,L.A. Confidential,Sympathy for Mr. Vengeance,La otra cara del crimen,Un grupo de personas planea un robo o estafa y...
6,U07,Sofía,definido,Velvet Goldmine,"Cuanto más, ¡mejor!",La vida de bohemia,Cero en conducta,Corazón salvaje,Una película sobre músicos o artistas que vive...
7,U08,Diego,definido,Superdetective en Hollywood,Mission: Impossible,Misión: Imposible 3,"Walker, Texas Ranger",300,Acción directa con un héroe que trabaja solo o...
8,U09,Elena,definido,Viaje a Darjeeling,Mi Idaho privado,Melinda y Melinda,La ciencia del sueño,Un beso,Algo tranquilo sobre personas que intentan rec...
9,U10,Facundo,definido,Sátántangó,Corazón salvaje,Mi Idaho privado,La ciencia del sueño,Sympathy for Mr. Vengeance,"Algo que sea difícil de clasificar, con una ló..."


Visualizamos las queries

In [6]:
for texto in usuarios['query']:
    print(texto)

Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Busco algo basado en hechos reales sobre corrupción o poder político
Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
No sé bien, algo que valga la pena ver un domingo a la noche, que enganche desde el princi

Construimos el dataset de peliculas, agregando un id que faltaba.

In [7]:
df_pelis["id"] = df_pelis.index + 1
df_pelis.head()

,description,keywords,genre,year,name,director,id
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak,1
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli,2
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben,3
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe,4
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich,5


## Preprocesado

### Función de limpieza de texto

Aplicamos un pipeline de limpieza estándar: minúsculas, eliminación de puntuación y palabras con números. 


In [8]:
def limpiar_texto(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)        # remover HTML si hubiera
    text = re.sub(r'[^\w\s\.,;:!?áéíóúüñ-]', ' ', text)  # caracteres extraños
    text = re.sub(r'\s+', ' ', text)             # espacios múltiples
    return text.strip()

Unificamos las variables relevantes en un texto (todas menos año)

In [9]:
df_pelis["texto"] = (
    df_pelis["name"].apply(limpiar_texto) + ". "
    + df_pelis["description"].apply(limpiar_texto) + ". "
    + df_pelis["director"].fillna('').apply(limpiar_texto) + ". "
    + df_pelis["genre"].apply(limpiar_texto) + ". "
    + df_pelis["keywords"].apply(limpiar_texto)
)

In [10]:
df_pelis.head()

,description,keywords,genre,year,name,director,id,texto
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak,1,"Herida abierta. Orin Boyd, un duro policía de ..."
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli,2,"Elvira, reina de las tinieblas. Al llegar a un..."
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben,3,Durmiendo con su enemigo. Una mujer finge su m...
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe,4,Elizabethtown. Durante un memorial en la ciuda...
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich,5,Godzilla. Las pruebas nucleares francesas irra...


#### El espanglish en ``genre`` y ``keywords``:  
El modelo multilingüe maneja texto en múltiples idiomas, pero fue entrenado con documentos monolingües por separado, no necesariamente con mezcla de idiomas dentro del mismo string

¿Es un problema grave? No. El modelo multilingüe es bastante robusto a esto. Pero sí es algo valioso para mencionar como limitación del corpus.

## Embedding de peliculas

In [11]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Calcular embedding de cada película

No hace falta hacer el promedio para calcularlo, el output del modelo ya es el embedding del documento (pelicula).

In [12]:
pelis_embeddings = model.encode(df_pelis['texto'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Como el dataset de películas y el de embeddings se construyeron en el mismo orden,podemos unirlos directamente por índice

In [13]:
df_pelis = df_pelis.reset_index(drop=True)

pelis_embeddings_df = pd.DataFrame(pelis_embeddings)
pelis_embeddings_df = pelis_embeddings_df.merge(
    df_pelis[['id', 'name']], 
    left_index=True, 
    right_index=True
)

In [14]:
pelis_embeddings_df.head()

,0,1,2,3,4,5,6,7,8,9,...,376,377,378,379,380,381,382,383,id,name
0,-0.005773,-0.000577,-0.285907,0.083817,0.095133,0.323502,0.153212,0.145991,0.061707,-0.164925,...,-0.077092,0.041387,-0.205363,-0.106746,-0.012276,-0.093761,0.219640,0.006689,1,Herida abierta
1,0.007110,0.052765,-0.072382,0.282494,-0.058281,-0.110369,0.350437,-0.005930,0.103852,0.027941,...,0.169869,0.247714,-0.010489,-0.118659,0.299659,-0.054366,-0.047600,-0.077431,2,"Elvira, reina de las tinieblas"
2,0.022418,0.007324,-0.113186,0.264627,0.084744,0.184328,0.088237,-0.082565,0.197784,-0.055697,...,0.149065,0.334282,-0.009026,0.114605,0.196578,0.048820,0.154733,-0.072756,3,Durmiendo con su enemigo
3,0.179020,-0.006517,-0.012674,0.009810,-0.112913,0.057386,0.169644,-0.142228,0.091899,0.070147,...,-0.029810,0.399069,0.038888,-0.121362,0.210644,0.105082,-0.001971,-0.047650,4,Elizabethtown
4,-0.151436,0.059364,-0.036458,-0.241015,0.113159,-0.308210,-0.030088,0.074282,0.057095,0.119314,...,-0.106445,-0.017355,-0.111064,0.232407,0.174773,-0.356135,0.414359,0.114730,5,Godzilla


## Embeddings de usuarios

Aplicamos la misma función de limpieza que usamos para las sinopsis. La verdad que no hace falta pero debería ser parte del pipeline operativo habitual.

In [15]:
# data_clean_users = pd.DataFrame(usuarios["query"].apply(limpiar_texto))

### Sin promediar

Con ``sentence_transformer`` no hace falta calcular los promedios de la query y el historial, se puede pasar todo el texto de una

el modelo tiene limite de 512 tokens, podemos chequear antes de calcular los embeddings.

In [16]:
# contar cantidad de palabras en df["texto"]
df_pelis["word_count"] = df_pelis["texto"].apply(lambda x: len(str(x).split()))
print("Cantidad maxima de palabras:")
print(df_pelis["word_count"].max())
print("Mediana de palabras:")
print(df_pelis["word_count"].median())

Cantidad maxima de palabras:
78
Mediana de palabras:
44.0


In [17]:
usuarios["query_word_count"] = usuarios["query"].apply(lambda x: len(str(x).split()))
print("Cantidad maxima de palabras en queries de usuarios:")
print(usuarios["query_word_count"].max())
print("Mediana de palabras en queries de usuarios:")
print(usuarios["query_word_count"].median())

Cantidad maxima de palabras en queries de usuarios:
19
Mediana de palabras en queries de usuarios:
15.5


Igualmente hay que considerar que las queries son cortas, podrían haber más largas en un futuro (quizás plantear una longitud máxima)

En promedio una palabra española se tokeniza en 1.5-2 tokens. Entonces:

``44 palabras × 5 películas + query (max 30 palabras, por ejemplo) ≈ 250 palabras ≈ 375-500 tokens``

Ejemplo exagerado del tokenizador

In [18]:
tokenizer = model.tokenizer

texto = "narcotraficante corruptísimo anticonstitucional"
tokens = tokenizer.tokenize(texto)
print(tokens)
print(f"palabras: 3 → tokens: {len(tokens)}")

['▁na', 'rc', 'otra', 'fica', 'nte', '▁', 'corrupt', 'ísimo', '▁antico', 'n', 'stitu', 'cional']
palabras: 3 → tokens: 12


In [19]:
def build_user_text(row, df_pelis):
    # Query del usuario
    partes = [row['query']]
    
    # Descripciones de las 5 películas del historial
    for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
        nombre = row[col]
        match = df_pelis[df_pelis['name'] == nombre]['texto']
        if len(match):
            partes.append(match.values[0])
    
    return ' '.join(partes)

user_texts = usuarios.apply(lambda r: build_user_text(r, df_pelis), axis=1).tolist()

In [20]:
user_embeddings = model.encode(user_texts, show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
user_embeddings_df = pd.DataFrame(user_embeddings)
user_embeddings_df = user_embeddings_df.merge(usuarios[['id']], left_index=True, right_index=True)
user_embeddings_df.head()

,0,1,2,3,4,5,6,7,8,9,...,375,376,377,378,379,380,381,382,383,id
0,-0.232565,-0.125751,-0.215482,0.231696,0.239823,0.302757,0.076590,-0.105773,0.208181,0.020862,...,0.328776,0.061258,0.363512,-0.099343,0.243902,0.132865,-0.136451,0.080300,-0.036140,U01
1,-0.279642,0.174792,-0.297886,0.230748,0.296417,0.116879,0.085544,-0.041812,0.023259,0.034446,...,-0.067355,-0.033141,0.187777,-0.215961,-0.111220,0.171769,-0.057322,-0.198638,-0.180328,U02
2,-0.016684,-0.304235,-0.068675,-0.139582,0.036174,0.338921,0.215532,0.029418,0.192602,-0.044089,...,0.043030,0.068193,0.000298,-0.330171,-0.049800,0.133548,0.114340,0.081853,0.023672,U03
3,-0.153277,-0.051864,-0.191920,-0.012805,0.022113,-0.185901,0.123533,0.043915,0.057784,0.108879,...,-0.048422,-0.111203,0.197444,-0.202105,0.130035,0.165597,0.043168,-0.044846,0.003998,U04
4,-0.013907,-0.004875,-0.237407,0.049726,0.151327,0.156037,0.255194,-0.196454,0.260561,0.152753,...,0.126489,0.143850,0.353202,-0.003771,0.190323,0.185765,-0.084386,-0.020892,-0.013917,U05


### Promediando

In [22]:
queries_embeddings = model.encode(usuarios['query'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
def get_all_historial_embeddings(usuarios_df, pelis_embeddings_df):
    all_embeddings = []
    
    for _, usuario_row in usuarios_df.iterrows():
        historial_embeddings = []
        
        for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
            nombre = usuario_row[col]
            peli_emb = pelis_embeddings_df[pelis_embeddings_df['name'] == nombre]
            if not peli_emb.empty:
                historial_embeddings.append(peli_emb.drop(columns=['id', 'name']).values[0])
            else:
                print(f"Película '{nombre}' no encontrada.")
        
        embedding_promedio = np.mean(historial_embeddings, axis=0)
        all_embeddings.append(embedding_promedio)
    
    return np.array(all_embeddings)

historial_embeddings = get_all_historial_embeddings(usuarios, pelis_embeddings_df)

Película 'Rec' no encontrada.
Película 'El secreto de sus ojos' no encontrada.
Película 'El exorcista' no encontrada.
Película 'Intocable' no encontrada.
Película 'Una mente brillante' no encontrada.
Película 'Paddington' no encontrada.


In [24]:
user_embeddings_2 = np.average(
    [queries_embeddings, historial_embeddings], 
    axis=0, 
    weights=[0.7, 0.3]
)

## Recomendaciones

### Sin promediar

In [25]:
scores = cosine_similarity(user_embeddings, pelis_embeddings)
# TODO: filtrar las películas que ya vio el usuario (historial) para no recomendarlas

# Top-10 por usuario
top10_indices = scores.argsort(axis=1)[:, -10:][:, ::-1]

# Ver resultados
for i, row in usuarios.iterrows():
    print(f"\n{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    for idx in top10_indices[i]:
        pelicula = df_pelis.iloc[idx]
        print(f"  {pelicula['name']} ({int(pelicula['year'])}) — {scores[i, idx]:.4f}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
  Durmiendo con su enemigo (1991) — 0.7553
  Tránsito (2006) — 0.7255
  Begotten (1991) — 0.6377
  Truly Madly Deeply (1992) — 0.6098
  Femme Fatale (2003) — 0.6055
  Melodía de seducción (1990) — 0.6026
  Pensamientos mortales (1991) — 0.5998
  Más allá de la muerte (2009) — 0.5959
  La novia cadáver (2005) — 0.5926
  The Ring (La señal) (2003) — 0.5873

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobre corrupción o poder político
  Adiós Bafana (2007) — 0.9485
  Historia de un soldado (1985) — 0.5755
  Una árida estación blanca (1989) — 0.5613
  Vida de este chico (1994) — 0.5609
  Monster's Ball (2002) — 0.5571
  Gorilas en la niebla (1989) — 0.5513
  Chopper (2001) — 0.5502
  El expreso de Elmira (2008) — 0.5498
  Rosewood (1997) — 0.5430
  Historia de un crimen (2007) — 0.5426

Camila (definido)
Query: Una comedia donde la relación entre dos

### Promediando

In [27]:
scores_2 = cosine_similarity(user_embeddings_2, pelis_embeddings)
# TODO: filtrar las películas que ya vio el usuario (historial) para no recomendarlas

# Top-10 por usuario
top10_indices_2 = scores_2.argsort(axis=1)[:, -10:][:, ::-1]

# Ver resultados
for i, row in usuarios.iterrows():
    print(f"\n{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    for idx in top10_indices_2[i]:
        pelicula = df_pelis.iloc[idx]
        print(f"  {pelicula['name']} ({int(pelicula['year']) if not pd.isna(pelicula['year']) else 'N/A'}) — {scores_2[i, idx]:.4f}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
  El ente (1983) — 0.6337
  Femme Fatale (2003) — 0.6297
  Tránsito (2006) — 0.6293
  Scary Movie 5 (2013) — 0.6033
  Luz de luna (1986) — 0.5937
  Alone in the Dark (2006) — 0.5882
  Conociendo a Julia (2004) — 0.5860
  Aquarius (1987) — 0.5840
  Inland Empire (2007) — 0.5834
  Fóllame (2001) — 0.5825

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobre corrupción o poder político
  L.A. Confidential (1997) — 0.5795
  El asesinato de Richard Nixon (2006) — 0.5680
  La noche cae sobre Manhattan (1997) — 0.5639
  Sospechoso (1988) — 0.5438
  Ciudad sin ley (2006) — 0.5418
  Negociador (1998) — 0.5356
  Dark Blue (2004) — 0.5353
  Enemigo público (1999) — 0.5352
  City Hall: La sombra de la corrupción (1996) — 0.5351
  Al filo de la muerte (2003) — 0.5347

Camila (definido)
Query: Una comedia donde la relación entre dos personas empieza de forma ridí

# Idea

Usar LLMs para analizar la query y decidir como funciona el sistema a partir de ella. Ejemplo:

query = "quiero ver algo distinto a lo de siempre"

con esa query el historial es más importante que la query en sí, pero no deberíamos buscar similaridad sino lo contrario

podríamos usar un LLM que reciba la query y tenga de outuput los pesos de la query e historial, más si debemos elegir las más similares o las menos similares.

el LLM debería c¿saber para que será usado